[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/templates/11_sliding_window.ipynb)

# 🔴 困难：滑动窗口注意力 (Sliding Window Attention)

实现 **滑动窗口注意力** —— 在 Longformer、Mistral 等模型中用于高效处理长上下文。

每个位置 $i$ 只能关注满足 $|i - j| \le w$ 的位置 $j$（其中 $w$ 为窗口大小）。

### 函数签名
```python
def sliding_window_attention(Q, K, V, window_size):
    # Q, K, V: (batch, seq, d) → 输出: (batch, seq, d_v)
    # window_size: int —— 位置 i 可关注 [i-w, i+w] 范围内的位置
```

### 要求
- **禁止** 使用稀疏注意力库
- 将窗口范围外的注意力位置用 `-inf` 进行掩码
- `window_size=0`：仅关注自身 —— 输出应等于 V
- `window_size >= seq_len`：等价于完整注意力（full attention）

In [ ]:
# 在 Colab 中安装 torch-judge（在 JupyterLab/Docker 中无操作）
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass


In [ ]:
import torch
import math

torch.arange(len, device) 创建一个顺序矩阵

distance = torch.abs(rows - cols) 创建

1. 计算完整的注意力得分
score[b, i, j] = Q[i] @ K[j].T

2. **创建两个 arange 矩阵，并得到注意力距离矩阵**
distance = torch.abs(rows - cols)

3. 对 distance 大于 window_size 的位置进行掩码，置 -∞

In [ ]:
# ✏️ 在此处实现你的代码

def sliding_window_attention(Q, K, V, window_size):
    # 获取维度信息
    # Q, K, V: (batch, seq, d)
    batch, seq_len, d = Q.shape
    
    # 1. 计算完整的注意力得分 (Attention Scores)
    # score[b, i, j] = Q[i] @ K[j].T
    # 复杂度: O(batch * seq_len^2 * d) -> 注意: [seq, d] * [d, seq] 计算复杂度为 seq ^ 2
    scores = torch.matmul(Q, K.transpose(-2, -1)) / (d ** 0.5)
    
    # 2. 处理 window_size=0 的特殊情况 (仅关注自身)
    if window_size == 0:
        # 在这种情况下，注意力矩阵应该是单位矩阵 I
        # 但更高效的方法是直接返回 V
        return V

    # 3. 创建滑动窗口掩码
    # 生成位置索引矩阵 [0, 1, ..., seq_len-1]
    cols = torch.arange(seq_len, device=Q.device).view(1, seq_len)
    rows = torch.arange(seq_len, device=Q.device).view(seq_len, 1)
    
    # 计算距离矩阵 |i - j|
    distance = torch.abs(rows - cols)
    
    # 构建掩码：当距离大于 window_size 时为 True
    mask = distance > window_size
    
    # 4. 应用掩码
    # 将超出窗口范围的位置设为负无穷，确保 Softmax 后权重为 0
    scores = scores.masked_fill(mask.unsqueeze(0), float('-inf'))
    
    # 5. 计算最终输出
    attn_weights = torch.softmax(scores, dim=-1) # (batch, seq, seq)
    output = torch.matmul(attn_weights, V)   # (batch, seq, d)
    
    return output

In [ ]:
# 🧪 调试测试
Q = torch.randn(1, 6, 8)
K = torch.randn(1, 6, 8)
V = torch.randn(1, 6, 8)

out = sliding_window_attention(Q, K, V, window_size=1)
print("输出形状:", out.shape)  # 应为 (1, 6, 8)

# window=0 时应返回 V
out0 = sliding_window_attention(Q, K, V, window_size=0)
print("window=0 时等于 V？", torch.allclose(out0, V, atol=1e-5))

In [ ]:
from torch_judge import check
check('sliding_window')